# Membrane RAG index builder
PDFs → evidence-carrying chunks (page / table / section) → hybrid BM25 + FAISS index.

**Before you start:** name the PDFs `P01.pdf`, `P02.pdf`, … so the ids match the Ground Truth.
Run the cells top to bottom. Read the *Parse health* and *Table spot-check* outputs before trusting the index.

**Streamlit app:** run section **9** at the end to export the files the Paper Research Agent app needs (`chunks.jsonl`, `embeddings.npy`, `embedding_info.json`, `pages.json`, `papers.json`).

In [ ]:
!pip install -q pymupdf sentence-transformers faiss-cpu rank_bm25 pandas openpyxl

In [ ]:
%%writefile membrane_index.py
"""
membrane_index.py  --  PDF -> evidence-carrying chunks -> hybrid (BM25 + dense/FAISS) retrieval.
Built for the Membrane Literature Comparison Agent. Field names follow the Extraction Schema.

Pipeline:  build_corpus(pdf_folder) -> Index.build(corpus, embedder) -> index.save(folder)
Backend :  Index.load(folder).search(...) / .retrieve_for_paper(pid) / .verify_snippet(...)
"""
import os, re, json, glob, math, zlib
from collections import Counter
import numpy as np
try:
    import fitz  # PyMuPDF -- only needed to PARSE pdfs; loading/searching a saved index does not need it
except ImportError:  # pragma: no cover
    fitz = None

# =====================================================================================
# 1. TEXT CLEANING
# =====================================================================================
_LIG = {"\ufb01": "fi", "\ufb02": "fl", "\ufb00": "ff", "\ufb03": "ffi", "\ufb04": "ffl",
        "\u00ad": "", "\u00a0": " ", "\u2009": " ", "\u202f": " ", "\u200b": ""}
KEEP_HYPHEN_PREFIX = {"cross", "dead", "non", "thin", "mixed", "pore", "self", "co", "anti", "semi",
                      "poly", "water", "dye", "solvent", "high", "low", "in", "ultra", "nano", "bi", "multi"}

def fix_chars(t):
    for a, b in _LIG.items():
        t = t.replace(a, b)
    return t

def join_lines(lines):
    """Join physical lines into a paragraph, repairing end-of-line hyphenation (rejec-/tion -> rejection)."""
    out = lines[0]
    for nxt in lines[1:]:
        m = re.search(r"([A-Za-z]{2,})-$", out)
        if m and nxt and nxt[0].islower():
            out = out + nxt if m.group(1).lower() in KEEP_HYPHEN_PREFIX else out[:-1] + nxt
        else:
            out = out + " " + nxt
    return re.sub(r"\s+", " ", out).strip()

def norm_match(t):
    """Aggressive normalisation used ONLY for matching/verification (never stored)."""
    t = fix_chars(t)
    t = t.translate(str.maketrans({"\u2212": "-", "\u2013": "-", "\u2014": "-", "\u2018": "'", "\u2019": "'",
                                   "\u201c": '"', "\u201d": '"', "\u207b": "-", "\u00b2": "2", "\u00b9": "1"}))
    t = re.sub(r"([A-Za-z]{2,})-\s+([a-z])", r"\1\2", t)          # leftover broken hyphenation
    return re.sub(r"\s+", " ", t).strip().lower()

# =====================================================================================
# 2. LAYOUT: blocks, headers/footers, columns, captions, sections
# =====================================================================================
class Block:
    __slots__ = ("text", "lines", "bbox", "size", "bold", "first_bold", "page")
    def __init__(self, text, lines, bbox, size, bold, first_bold, page):
        self.text, self.lines, self.bbox, self.size, self.bold, self.first_bold, self.page = \
            text, lines, bbox, size, bold, first_bold, page

def page_blocks(page, pno):
    out = []
    for b in page.get_text("dict")["blocks"]:
        if b.get("type") != 0: continue
        lines, sizes, bolds = [], [], []
        for l in b["lines"]:
            txt = fix_chars("".join(s["text"] for s in l["spans"])).strip()
            if not txt: continue
            lines.append(txt)
            for s in l["spans"]:
                if s["text"].strip():
                    sizes.append(s["size"]); bolds.append(bool(s["flags"] & 16) or "bold" in s["font"].lower())
        if not lines: continue
        if is_junk(join_lines(lines)) and len(lines) > 0 and not CAPTION_RE.match(lines[0]): continue
        out.append(Block(join_lines(lines), lines, tuple(b["bbox"]), max(sizes), sum(bolds) / len(bolds) > 0.6, bolds[0], pno))
    return out

def is_junk(text):
    t = text.strip(); n = len(t.replace(" ", ""))
    if n == 0: return True
    letters, digits = sum(c.isalpha() for c in t), sum(c.isdigit() for c in t)
    if letters / n < 0.45 and digits / n < 0.35: return True                       # symbol soup (broken font maps)
    toks = t.split()
    if len(toks) >= 8 and sum(len(w) <= 2 for w in toks) / len(toks) > 0.5 and not re.search(r"[.;:]\s", t): return True   # axis / figure labels
    return False

def order_blocks(blocks, W):
    """Reading order. Detects 2-column pages and reads left column then right (between full-width blocks)."""
    if not blocks: return []
    mid = W / 2
    def side(b):
        x0, _, x1, _ = b.bbox
        return 0 if x1 <= mid + 8 else (1 if x0 >= mid - 8 else 2)
    sides = {id(b): side(b) for b in blocks}
    nl = sum(1 for b in blocks if sides[id(b)] == 0 and len(b.text) > 25)
    nr = sum(1 for b in blocks if sides[id(b)] == 1 and len(b.text) > 25)
    if nl >= 2 and nr >= 2:
        full = sorted([b for b in blocks if sides[id(b)] == 2], key=lambda b: b.bbox[1])
        rest = [b for b in blocks if sides[id(b)] != 2]
        ordered = []
        for i in range(len(full) + 1):
            lo = full[i - 1].bbox[3] if i > 0 else -1e9
            hi = full[i].bbox[1] if i < len(full) else 1e9
            seg = [b for b in rest if lo <= (b.bbox[1] + b.bbox[3]) / 2 < hi]
            ordered += sorted([b for b in seg if sides[id(b)] == 0], key=lambda b: b.bbox[1])
            ordered += sorted([b for b in seg if sides[id(b)] == 1], key=lambda b: b.bbox[1])
            if i < len(full): ordered.append(full[i])
        return ordered
    return sorted(blocks, key=lambda b: (round(b.bbox[1] / 3), b.bbox[0]))

_pagenum = re.compile(r"\d+")
def strip_headers_footers(pages_blocks, heights):
    """Remove running headers/footers/page numbers. Returns (clean_pages, removed_texts)."""
    n = len(pages_blocks)
    sig = Counter()
    for blocks, H in zip(pages_blocks, heights):
        for b in blocks:
            if b.bbox[3] < 0.12 * H or b.bbox[1] > 0.88 * H:
                sig[_pagenum.sub("#", norm_match(b.text))] += 1
    removed, cleaned = [], []
    for blocks, H in zip(pages_blocks, heights):
        sizes = sorted(b.size for b in blocks) or [10]
        body = sizes[len(sizes) // 2]
        keep = []
        for b in blocks:
            in_top, in_bot = b.bbox[3] < 0.065 * H, b.bbox[1] > 0.935 * H
            repeated = n >= 2 and sig[_pagenum.sub("#", norm_match(b.text))] >= max(2, 0.4 * n) \
                       and (b.bbox[3] < 0.12 * H or b.bbox[1] > 0.88 * H)
            short_edge = (in_top or in_bot) and len(b.text) < 200 and b.size <= 1.25 * body
            if repeated or short_edge: removed.append(b.text)
            else: keep.append(b)
        cleaned.append(keep)
    return cleaned, removed

CAPTION_RE = re.compile(r"^\s*(Table|TABLE|Fig\.?|FIG\.?|Figure|FIGURE|Scheme)\s*(S?\d+[A-Za-z]?)\s*(?P<p>[.:|\u2013\u2014\-]?)\s*(?P<rest>.*)$", re.S)
_VERBS = re.compile(r"^\s*(lists?|shows?|summari[sz]es?|presents?|gives?|compares?|illustrates?|depicts?|displays?|reports?|provides?|indicates?|demonstrates?|and|of|in|is|are|was|were)\b", re.I)

def caption_of(b):
    """Return (kind, label, text) if the block is a table/figure caption, else None."""
    m = CAPTION_RE.match(b.text)
    if not m: return None
    p, rest = m.group("p"), m.group("rest").strip()
    if not rest and not p: pass
    elif p in (".", ":", "|") or b.first_bold:
        pass
    elif len(b.text) < 350 and rest[:1].isupper() and not _VERBS.match(rest):
        pass
    else:
        return None
    if p == "" and _VERBS.match(rest): return None
    kind = "Table" if m.group(1).lower().startswith("t") else ("Figure" if m.group(1).lower().startswith("f") else "Scheme")
    label = f"{kind} {m.group(2)}"
    return kind, label, f"{label}. {rest}" if rest else label

SECTION_MAP = [
    ("abstract", r"abstract|summary"),
    ("keywords", r"key\s*words?|highlights"),
    ("introduction", r"introduction|background"),
    ("methods", r"materials?\s+and\s+methods?|experimental(?:\s+(?:section|procedures?|methods?|details?|part))?|methods?|methodology|materials?|experiments?"),
    ("results", r"results?(?:\s+and\s+discussions?)?|results?\s*&\s*discussions?|discussions?"),
    ("conclusion", r"conclusions?(?:\s+and\s+(?:outlook|future\s+(?:work|perspectives?)))?|concluding\s+remarks"),
    ("acknowledgements", r"acknowledge?ments?|funding|declarations?|credit\s+authorship.*|conflicts?\s+of\s+interest|author\s+contributions?"),
    ("references", r"references|bibliography|literature\s+cited"),
    ("supplementary", r"supplementary.*|supporting\s+information|appendix.*"),
]
_SEC_RES = [(g, re.compile(r"^\s*(?:\d+(?:\.\d+)*\.?\s+|[IVX]+\.\s+)?(?:%s)\s*[:.]?\s*$" % rx, re.I)) for g, rx in SECTION_MAP]
_RUNIN = re.compile(r"^\s*(abstract|keywords?)\s*[:.\u2014\u2013-]\s*(.+)$", re.I | re.S)
_SUBHEAD = re.compile(r"^\s*\d+(?:\.\d+)+\.?\s+\S.{2,90}$")
_TOPHEAD = re.compile(r"^\s*(?:\d{1,2}|[IVX]{1,4})\.?\s+[A-Z][^.!?]{2,90}$")
def infer_group(h):
    x = h.lower()
    if re.search(r"result|discussion|performance|evaluation|analysis|comparison|effects? of", x): return "results"
    if re.search(r"experiment|method|material|preparation|fabrication|synthesis|procedure|set-?up|apparatus|measurement|testing", x): return "methods"
    if re.search(r"related work|background|literature|motivation|overview", x): return "introduction"
    return "other"

def spaced_fix(t):
    t = t.strip()
    return re.sub(r"\s", "", t) if re.fullmatch(r"(?:[A-Za-z]\s){3,}[A-Za-z]", t) else t

def classify_heading(b, body_size):
    """-> ('section', group, rest_text) | ('sub', heading_text) | None"""
    t = spaced_fix(b.text)
    if len(b.lines) <= 2 and len(t) < 90:
        for g, rx in _SEC_RES:
            if rx.match(t): return ("section", g, "")
    m = _RUNIN.match(b.text)
    if m and (b.first_bold or b.text[len(m.group(1)):len(m.group(1)) + 1] in ".:\u2014\u2013-"):
        return ("section", "abstract" if m.group(1).lower() == "abstract" else "keywords", m.group(2).strip())
    if len(b.lines) <= 2 and len(t) < 100 and _SUBHEAD.match(t) and (b.bold or b.size > body_size * 1.02):
        return ("sub", t)
    if len(b.lines) <= 2 and len(t) < 100 and _TOPHEAD.match(t) and (b.bold or b.size > body_size * 1.02):
        return ("top", infer_group(t), t)
    return None

# =====================================================================================
# 3. TABLES  (ruled tables via PyMuPDF + three-line/borderless tables via horizontal rules)
# =====================================================================================
_NUM = re.compile(r"^\(?[<>\u2248~\u00b1+\-]?\s*\d+([.,]\d+)?(\s*(\u00b1|\+/-)\s*\d+([.,]\d+)?)?\s*%?\)?$")
def is_num(c): return bool(c) and bool(_NUM.match(c.strip()))

def n_header_rows(rows):
    n = 0
    for r in rows[:4]:
        cells = [c for c in r[1:] if c and c.strip()]
        frac = (sum(is_num(c) for c in cells) / len(cells)) if cells else 0
        if cells and all(re.fullmatch(r"(19|20)\d{2}", c.strip()) for c in cells): frac = 0      # a row of years is a header
        if frac >= 0.5: break
        n += 1
    if n >= len(rows): return 1          # no numeric data at all: assume one header row
    return n                             # n == 0 -> table has no detectable header (image header?) -> generic col names

def flatten_header(hrows, ncols):
    """Spanning headers are forward-filled; multi-row headers are merged per column."""
    filled = []
    for ri, r in enumerate(hrows):
        r = list(r) + [None] * (ncols - len(r)); out = list(r)
        if ri < len(hrows) - 1:
            lower = hrows[ri + 1] + [None] * (ncols - len(hrows[ri + 1]))
            for c in range(1, ncols):
                if r[c] is None and out[c - 1] and (lower[c] not in (None, "")):
                    out[c] = out[c - 1]
        filled.append(out)
    names = []
    for c in range(ncols):
        parts = []
        for r in filled:
            v = (r[c] or "").strip()
            if v and (not parts or parts[-1] != v): parts.append(v)
        s = ""
        for p in parts:
            s = p if not s else (s + " " + p if p[0] in "([" else s + " / " + p)
        names.append(s or f"col{c+1}")
    return names

def _clean_cell(c): return re.sub(r"\s+", " ", fix_chars(c)).strip() if isinstance(c, str) else c

def ruled_tables(page):
    out = []
    try:
        for t in page.find_tables().tables:
            rows = [[_clean_cell(c) for c in r] for r in t.extract()]
            if len(rows) >= 1 and len(rows[0]) >= 2 and any(any(c for c in r if c) for r in rows):
                out.append({"bbox": tuple(t.bbox), "rows": rows, "kind": "ruled"})
    except Exception:
        pass
    return out

def _h_rules(page):
    rules = []
    for d in page.get_drawings():
        for it in d["items"]:
            if it[0] == "l":
                p1, p2 = it[1], it[2]
                if abs(p1.y - p2.y) < 1.2 and abs(p1.x - p2.x) > 50: rules.append((min(p1.x, p2.x), max(p1.x, p2.x), (p1.y + p2.y) / 2))
            elif it[0] == "re":
                r = it[1]
                if r.height < 2.2 and r.width > 50: rules.append((r.x0, r.x1, (r.y0 + r.y1) / 2))
    rules.sort(key=lambda r: r[2]); ded = []
    for r in rules:
        if ded and abs(ded[-1][2] - r[2]) < 1.5 and abs(ded[-1][0] - r[0]) < 6: continue
        ded.append(r)
    return ded

def three_line_tables(page, caption_blocks, taken):
    """Tables with only horizontal rules (the dominant journal style)."""
    rules = _h_rules(page); groups = []
    for r in rules:
        if groups:
            g = groups[-1]; gx0, gx1 = min(q[0] for q in g), max(q[1] for q in g)
            contained = r[0] >= gx0 - 12 and r[1] <= gx1 + 12          # partial rules (cmidrules under spanning headers) belong to the table
            cap_between = any(g[-1][2] < (cb.bbox[1] + cb.bbox[3]) / 2 < r[2] for cb in caption_blocks)
            if contained and not cap_between and r[2] - g[-1][2] < 400:
                g.append(r); continue
        groups.append([r])
    words = page.get_text("words"); out = []
    for g in groups:
        if len(g) < 2: continue
        x0, x1, ytop, ybot = min(r[0] for r in g), max(r[1] for r in g), g[0][2], g[-1][2]
        if any(not (ybot < t[1] or ytop > t[3] or x1 < t[0] or x0 > t[2]) for t in taken): continue
        ws = [w for w in words if w[1] >= ytop - 1 and w[3] <= ybot + 1 and w[0] >= x0 - 2 and w[2] <= x1 + 2]
        if len(ws) < 4: continue
        # rows
        ws.sort(key=lambda w: (w[1] + w[3]) / 2); rows = []
        for w in ws:
            cy = (w[1] + w[3]) / 2
            if rows and abs(cy - rows[-1]["cy"]) < 0.55 * (w[3] - w[1]): rows[-1]["w"].append(w)
            else: rows.append({"cy": cy, "w": [w]})
        if len(rows) < 2: continue
        interior = g[1:-1]; full = [q for q in interior if (q[1] - q[0]) >= 0.9 * (x1 - x0)]
        sep_y = max(q[2] for q in full) if full else (interior[0][2] if interior else None)
        hdr_idx = [i for i, r in enumerate(rows) if sep_y and r["cy"] < sep_y]
        data_rows = [r for i, r in enumerate(rows) if i not in hdr_idx] or rows
        # columns from data rows (gaps in x-projection)
        ivs = sorted((w[0], w[2]) for r in data_rows for w in r["w"]); cols = []
        for a, b in ivs:
            if cols and a - cols[-1][1] < 5.0: cols[-1][1] = max(cols[-1][1], b)
            else: cols.append([a, b])
        if len(cols) < 2: continue
        def col_span(a, b): return [i for i, (ca, cb) in enumerate(cols) if min(b, cb) - max(a, ca) > 0]
        table = []
        for ri, r in enumerate(rows):
            cells = [None if ri in hdr_idx else ""] * len(cols)
            phrases = []
            for w in sorted(r["w"], key=lambda w: w[0]):
                if phrases and w[0] - phrases[-1][2] < 5.0: phrases[-1][2] = w[2]; phrases[-1][4] += " " + w[4]
                else: phrases.append([w[0], w[1], w[2], w[3], w[4]])
            for p in phrases:
                span = col_span(p[0], p[2]) or [min(range(len(cols)), key=lambda i: abs((cols[i][0] + cols[i][1]) / 2 - (p[0] + p[2]) / 2))]
                if ri in hdr_idx:
                    for c in span: cells[c] = p[4] if cells[c] in (None, "") else cells[c] + " " + p[4]
                else:
                    c = span[0]; cells[c] = (cells[c] + " " if cells[c] else "") + p[4]
            table.append([_clean_cell(c) if c is not None else None for c in cells])
        out.append({"bbox": (x0, ytop, x1, ybot), "rows": table, "kind": "three-line"})
    return out

def split_header_data(rows):
    nh = n_header_rows([[("" if c is None else c) for c in r] for r in rows])
    ncols = max(len(r) for r in rows)
    header = flatten_header([list(r) + [None] * (ncols - len(r)) for r in rows[:nh]], ncols)
    data = [[("" if c is None else str(c)) for c in list(r) + [None] * (ncols - len(r))] for r in rows[nh:]]
    data = [r for r in data if any(c.strip() for c in r)]
    return header, data

# =====================================================================================
# 4. CHUNKING
# =====================================================================================
_ABBR = ["et al.", "Fig.", "Figs.", "Eq.", "Eqs.", "vs.", "ca.", "approx.", "Ref.", "No.", "e.g.", "i.e.", "Dr.", "wt.", "vol.", "cf."]
def split_sentences(p):
    for a in _ABBR: p = p.replace(a, a.replace(".", "\u2024"))
    p = re.sub(r"(\d)\.(\d)", "\\1\u2024\\2", p)
    s = re.split(r"(?<=[.!?])\s+(?=[A-Z\[(0-9])", p)
    return [x.replace("\u2024", ".") for x in s if x.strip()]

def chunk_paragraphs(paras, size, overlap):
    sents = split_sentences(join_lines(paras)) if paras else []
    chunks, cur, n, fresh = [], [], 0, False
    for s in sents:
        w = len(s.split())
        if cur and n + w > size and fresh:
            chunks.append(" ".join(cur)); keep, m = [], 0
            for t in reversed(cur):
                if m + len(t.split()) > overlap: break
                keep.insert(0, t); m += len(t.split())
            cur, n, fresh = keep, m, False
        cur.append(s); n += w; fresh = True
    if cur and fresh: chunks.append(" ".join(cur))
    return chunks

QUANT_RE = re.compile(r"\d+(?:\.\d+)?\s*(?:%|lmh|bar|kpa|mpa|psi|mg\s*/\s*l|g\s*/\s*l|ppm|wt\s*%|nm|\u00b5m|\u03bcm|\u00b0c|min|h\b|barrer|gpu)", re.I)

TAGS_CI = {
 "metric": r"permeate flux|water flux|pure water flux|permeation flux|\bflux\b|rejection|retention|removal efficiency|dye removal|separation efficiency|permeance|permeability|selectivity|separation factor|recovery",
 "polymer": r"polysulfone|polyethersulfone|polyvinylidene|polyacrylonitrile|polyamide|cellulose acetate|polyimide|polybenzimidazole|\budel\b|kynar|matrimid",
 "filler": r"metal-organic framework|zif-?\d*|mwcnt|swcnt|carbon nanotube|graphene oxide|zeolite|zsm-5|titania|tio2|silver nanoparticle",
 "test_mode": r"dead-end|cross-?flow|stirred cell|submerged|tangential",
 "pressure_type": r"transmembrane pressure|applied pressure|feed pressure",
 "config": r"flat sheet|hollow fib(?:er|re)|spiral wound|tubular",
 "scale": r"lab-scale|bench-scale|pilot|industrial",
 "membrane_type": r"nanofiltration|ultrafiltration|microfiltration|reverse osmosis|forward osmosis|membrane distillation|gas separation|pervaporation|mixed-matrix|thin-film (?:composite|nanocomposite)",
 "unit": r"\blmh\b|\bgpu\b|barrer|\bkpa\b|\bmpa\b|\bpsi\b|\bbar\b|mg/l|\bppm\b",
}
TAGS_CS = {"abbr": r"\b(MF|UF|NF|RO|FO|MD|MMM|TFC|TFN|PSf|PES|PVDF|PAN|MOF|ZIF|CNT|GO|rGO|TMP|MWCO)\b"}
def tag_text(t):
    tags = set()
    for k, rx in TAGS_CI.items():
        for m in re.findall(rx, t, flags=re.I): tags.add(f"{k}:{m.lower()}")
    for k, rx in TAGS_CS.items():
        for m in re.findall(rx, t): tags.add(f"{k}:{m}")
    return sorted(tags)

# =====================================================================================
# 5. PAPER METADATA
# =====================================================================================
KNOWN_JOURNALS = ["Journal of Membrane Science", "Separation and Purification Technology", "Desalination", "Water Research",
    "Chemical Engineering Journal", "Journal of Hazardous Materials", "Journal of Water Process Engineering", "Membranes",
    "Polymers", "Scientific Reports", "Chemosphere", "Applied Surface Science", "Environmental Science & Technology",
    "ACS Applied Materials & Interfaces", "Journal of Environmental Chemical Engineering", "Process Safety and Environmental Protection",
    "Journal of Cleaner Production", "Nanomaterials", "Water", "RSC Advances", "Journal of Applied Polymer Science"]
_AFFIL = re.compile(r"department|university|institute|school|college|laborator|faculty|centre|center|@|\bP\.?O\.? Box", re.I)

def detect_year(front, doi):
    pri = [r"published[^\d]{0,30}(?:\d{1,2}\s+\w+\s+)?((?:19|20)\d{2})", r"available online[^\d]{0,30}(?:\d{1,2}\s+\w+\s+)?((?:19|20)\d{2})",
           r"accepted[^\d]{0,30}(?:\d{1,2}\s+\w+\s+)?((?:19|20)\d{2})", r"(?:\u00a9|\(c\)|copyright)\s*((?:19|20)\d{2})",
           r"\(((?:19|20)\d{2})\)\s*\|"]
    for rx in pri:
        m = re.search(rx, front, re.I)
        if m: return int(m.group(1))
    if doi:
        m = re.search(r"[./-]((?:19|20)\d{2})[./-]", doi)
        if m: return int(m.group(1))
    ys = [int(y) for y in re.findall(r"\b((?:199\d|20[0-2]\d))\b", front) if int(y) <= 2026]
    return Counter(ys).most_common(1)[0][0] if ys else None

def detect_meta(pid, path, pages_blocks_raw, ordered_p1, removed, doc, source_type):
    p1 = ordered_p1
    body = sorted(b.size for b in p1)[len(p1) // 2] if p1 else 10
    title, authors = None, None
    if p1:
        mx = max(b.size for b in p1)
        tb = [b for b in p1 if b.size >= 0.92 * mx and len(b.text) > 8 and classify_heading(b, body) is None]
        if tb:
            tb.sort(key=lambda b: b.bbox[1]); grp = tb[:1]
            for b in tb[1:]:
                if -8 <= b.bbox[1] - grp[-1].bbox[3] < 25: grp.append(b)
            title = re.sub(r"\s+", " ", " ".join(b.text for b in grp)).strip()
            after = [b for b in p1 if b.bbox[1] >= grp[-1].bbox[3] - 1 and b not in grp]
            after.sort(key=lambda b: (b.bbox[1], b.bbox[0]))
            def _is_authors(ln):
                if _AFFIL.search(ln) or len(ln) > 250 or ln.lower().startswith("abstract") or ln.rstrip().endswith("."): return False
                if "," in ln or " and " in ln: return True
                return bool(re.match(r"^[A-Z][\w.\-]*\.?(\s+[A-Z][\w.\-]*\.?){1,3}\s*[\d*,a-z]{0,6}$", ln)) and len(ln.split()) <= 5
            for b in after[:3]:
                if classify_heading(b, body): break
                got = []
                for ln in b.lines[:6]:
                    if _is_authors(ln): got.append(ln)
                    elif got: break
                if got: authors = " ".join(got); break
    raw1 = " ".join(removed[:6]) + " " + " ".join(b.text for b in p1)
    doi_m = re.search(r"10\.\d{4,9}/[^\s\"<>]+", raw1)
    doi = doi_m.group(0).rstrip(".,;)") if doi_m else None
    hf = Counter(_pagenum.sub("#", re.sub(r"\s+", " ", t)) for t in removed)
    journal = None
    for j in sorted(KNOWN_JOURNALS, key=len, reverse=True):
        if re.search(r"\b%s\b" % re.escape(j), " ".join(hf) + " " + raw1[:1500], re.I): journal = j; break
    if not journal and hf:
        cand = hf.most_common(1)[0][0]; cand = re.split(r"\s*[|\u00b7\u2022]\s*", cand)[0].strip()
        if 3 < len(cand) < 80 and not re.match(r"^(page|#)", cand, re.I): journal = cand
    return {"paper_id": pid, "title": title, "authors": authors, "publication_year": detect_year(raw1, doi), "journal": journal,
            "doi": doi, "source_url": None, "source_type": source_type, "file": os.path.basename(path), "num_pages": len(doc),
            "needs_review": [k for k, v in (("title", title), ("authors", authors), ("journal", journal), ("publication_year", detect_year(raw1, doi)), ("doi", doi)) if not v] +
                            ["authors/journal are heuristic: verify"], "header_footer": [t for t, _ in hf.most_common(3)]}

# =====================================================================================
# 6. PER-PAPER PROCESSING
# =====================================================================================
def process_pdf(path, pid, source_type="PDF upload", chunk_words=220, overlap_words=40, max_table_rows=25):
    if fitz is None: raise ImportError("PyMuPDF is required to parse PDFs: pip install pymupdf")
    doc = fitz.open(path)
    raw_pages, heights, widths = [], [], []
    for pno in range(len(doc)):
        pg = doc[pno]; raw_pages.append(page_blocks(pg, pno + 1)); heights.append(pg.rect.height); widths.append(pg.rect.width)
    clean_pages, removed = strip_headers_footers(raw_pages, heights)
    ordered = [order_blocks(b, w) for b, w in zip(clean_pages, widths)]
    meta = detect_meta(pid, path, raw_pages, ordered[0] if ordered else [], removed, doc, source_type)
    body_sizes = sorted(b.size for pg in ordered for b in pg) or [10]; body = body_sizes[len(body_sizes) // 2]

    chunks, tables_out, page_text = [], [], {}
    state = {"section": "front_matter", "heading": ""}
    prev_table = None   # for continuation across pages
    for pno, blocks in enumerate(ordered):
        page = doc[pno]; page_no = pno + 1
        caps = [(b, caption_of(b)) for b in blocks]; caps = [(b, c) for b, c in caps if c]
        tabs = ruled_tables(page)
        tabs += three_line_tables(page, [b for b, c in caps if c[0] == "Table"], [t["bbox"] for t in tabs])
        def assoc(t):
            x0, y0, x1, y1 = t["bbox"]
            for kind in ("Table", "Figure"):
                above = [(b, c) for b, c in caps if c[0] == kind and b.bbox[3] <= y0 + 4 and y0 - b.bbox[3] < 70 and b.bbox[0] < x1 and b.bbox[2] > x0]
                below = [(b, c) for b, c in caps if c[0] == kind and b.bbox[1] >= y1 - 4 and b.bbox[1] - y1 < 40 and b.bbox[0] < x1 and b.bbox[2] > x0]
                if above: return max(above, key=lambda bc: bc[0].bbox[3])
                if below: return below[0]
            return None
        def numfrac(t):
            _, data = split_header_data(t["rows"]); cells = [c for r in data for c in r[1:] if c.strip()]
            return (sum(is_num(c) for c in cells) / len(cells)) if cells else 0.0
        kept = []
        for t in tabs:
            pk = assoc(t); t["pick"] = pk; nf = numfrac(t); nrows = len(split_header_data(t["rows"])[1])
            cont = bool(prev_table and pno > 0 and t["bbox"][1] < 0.3 * heights[pno] and prev_table["page"] == page_no - 1 and len(t["rows"][0]) == len(prev_table["header"]))
            if pk and pk[1][0] == "Table": ok = len(t["rows"]) >= 2 or cont
            elif pk: ok = nf >= 0.5 and nrows >= 3          # numeric grid captioned as a Figure (e.g. "Figure 13. statistics")
            else: ok = cont or (nf >= 0.3 and nrows >= 2)
            if ok: kept.append(t)
        tabs = sorted(kept, key=lambda t: t["bbox"][1])
        def in_table(b):
            cx, cy = (b.bbox[0] + b.bbox[2]) / 2, (b.bbox[1] + b.bbox[3]) / 2
            return any(t["bbox"][0] - 3 <= cx <= t["bbox"][2] + 3 and t["bbox"][1] - 3 <= cy <= t["bbox"][3] + 3 for t in tabs)
        # walk blocks in order -> sections, text groups, captions
        groups, sec_at = [], {}
        sec_start = dict(state)
        for b in blocks:
            if in_table(b): continue
            h = classify_heading(b, body)
            cap = caption_of(b)
            if h and h[0] == "section":
                state["section"], state["heading"] = h[1], ""
                if h[2]: groups.append([state["section"], "", [h[2]]])
                sec_at[id(b)] = dict(state); continue
            if h and h[0] == "sub":
                state["heading"] = h[1]; sec_at[id(b)] = dict(state); continue
            if h and h[0] == "top":
                state["section"], state["heading"] = h[1], h[2]; sec_at[id(b)] = dict(state); continue
            sec_at[id(b)] = dict(state)
            if cap: continue
            if groups and groups[-1][0] == state["section"] and groups[-1][1] == state["heading"]: groups[-1][2].append(b.text)
            else: groups.append([state["section"], state["heading"], [b.text]])
        page_text[page_no] = " ".join(norm_match(b.text) for b in blocks) + " " + " ".join(norm_match(" ".join(" ".join(c or "" for c in r) for r in t["rows"])) for t in tabs)
        for sec, head, paras in groups:
            for ch in chunk_paragraphs(paras, chunk_words, overlap_words):
                chunks.append(dict(paper_id=pid, evidence_page=page_no, section=sec, heading=head, chunk_type="text", evidence_type="Text",
                                   evidence_location=f"Section: {sec}" + (f" > {head}" if head else ""), table_index=None, text=ch))
        used_caps = set()
        for ti, t in enumerate(tabs):
            x0, y0, x1, y1 = t["bbox"]
            pick = t.get("pick")
            header, data = split_header_data(t["rows"])
            continued = False
            if pick:
                used_caps.add(id(pick[0])); label, caption = pick[1][1], pick[1][2]
            elif prev_table and pno > 0 and len(header) == len(prev_table["header"]) and y0 < 0.3 * heights[pno] and prev_table["page"] == page_no - 1:
                label, caption = prev_table["label"], prev_table["caption"] + " (continued)"; header = prev_table["header"]
                data = [[("" if c is None else str(c)) for c in r] for r in t["rows"] if any(c for c in r)]; continued = True
            else:
                label, caption = f"Table (page {page_no}, #{ti+1})", ""
            if not data and not continued: continue
            # section for the table = section of nearest block above
            above = [b for b in blocks if id(b) in sec_at and b.bbox[3] <= y0 + 3 and b.bbox[0] < x1 and b.bbox[2] > x0]
            ssec = sec_at[id(max(above, key=lambda b: b.bbox[3]))] if above else sec_start
            for ci in range(0, max(len(data), 1), max_table_rows):
                part = data[ci:ci + max_table_rows]
                lines = [caption or label, "Columns: " + " | ".join(header)] + [" | ".join(r) for r in part]
                split = len(data) > max_table_rows
                loc = f"{label} (rows {ci + 1}-{ci + len(part)})" if split else label   # row-level locator for long tables
                chunks.append(dict(paper_id=pid, evidence_page=page_no, section=ssec["section"], heading=ssec["heading"], chunk_type="table",
                                   evidence_type="Figure" if label.startswith("Figure") else "Table", evidence_location=loc, table_index=ti, text="\n".join(lines),
                                   table={"label": label, "caption": caption, "header": header, "rows": part, "continued": continued, "detector": t["kind"],
                                          "row_start": ci + 1, "row_end": ci + len(part), "n_rows_total": len(data)}))
            prev_table = {"label": label, "caption": caption, "header": header, "page": page_no}
        for b, c in caps:
            if id(b) in used_caps: continue
            if c[0] == "Table" and any(not (ch is None) for ch in []): continue
            sc = sec_at.get(id(b), sec_start)
            chunks.append(dict(paper_id=pid, evidence_page=page_no, section=sc["section"], heading=sc["heading"], chunk_type="caption",
                               evidence_type="Caption" if c[0] != "Figure" else "Figure", evidence_location=c[1], table_index=None, text=c[2]))
    meta["front_matter"] = " ".join(removed[:3] + [b.text for b in (ordered[0][:6] if ordered else [])])[:1500]
    return meta, chunks, page_text

def build_corpus(pdf_folder, source_type="PDF upload", chunk_words=220, overlap_words=40):
    paths = sorted(glob.glob(os.path.join(pdf_folder, "*.pdf")))
    ids = {}
    for p in paths:                                     # pass 1: files named P<number>*.pdf  (P1.pdf -> P01 so ids match the Ground Truth)
        m = re.match(r"^P0*(\d+)", os.path.basename(p), re.I)
        if m:
            pid = f"P{int(m.group(1)):02d}"
            assert pid not in ids.values(), f"duplicate paper id {pid} ({os.path.basename(p)}): name PDFs P01.pdf, P02.pdf ..."
            ids[p] = pid
    nxt = 1 + max([int(v[1:]) for v in ids.values()] or [0])
    for p in paths:                                     # pass 2: other names get the next FREE id (never collides) and a warning
        if p not in ids:
            ids[p] = f"P{nxt:02d}"; nxt += 1
            print(f"[warn] {os.path.basename(p)} is not named P<number>.pdf; assigned id {ids[p]} -- rename it so ids match the Ground Truth")
    papers, chunks, pages = [], [], {}
    for p in paths:
        pid = ids[p]
        meta, ch, pt = process_pdf(p, pid, source_type, chunk_words, overlap_words)
        papers.append(meta); chunks += ch; pages[pid] = pt
    pmap = {p["paper_id"]: p for p in papers}
    for i, c in enumerate(chunks):
        p = pmap[c["paper_id"]]
        c.update(id=i, title=p["title"], doi=p["doi"], publication_year=p["publication_year"], source_type=p["source_type"],
                 preview=c["text"][:200].replace("\n", " "), word_count=len(c["text"].split()), tags=tag_text(c["text"]), has_quantity=bool(QUANT_RE.search(c["text"])))
    return {"papers": papers, "chunks": chunks, "pages": pages}

def parse_health(corpus):
    rows = []
    for p in corpus["papers"]:
        pc = [c for c in corpus["chunks"] if c["paper_id"] == p["paper_id"]]
        textp = {c["evidence_page"] for c in pc if c["chunk_type"] == "text"}; flags = []
        if len(textp) < 0.7 * p["num_pages"]: flags.append("LOW TEXT (scanned PDF? needs OCR)")
        if not any(c["chunk_type"] == "table" for c in pc): flags.append("NO TABLES FOUND")
        if not any(c["section"] == "abstract" for c in pc): flags.append("NO ABSTRACT")
        if not any(c["section"] == "results" for c in pc): flags.append("NO RESULTS SECTION")
        for k in ("title", "doi", "publication_year", "journal", "authors"):
            if not p.get(k): flags.append(f"NO {k.upper()}")
        orphan = [c["evidence_location"] for c in pc if c["chunk_type"] == "caption" and c["evidence_location"].startswith("Table")]
        if orphan: flags.append(f"CAPTION WITHOUT PARSED TABLE (image table? check manually): {', '.join(orphan)}")
        unl = [c["evidence_location"] for c in pc if c["chunk_type"] == "table" and c["evidence_location"].startswith("Table (page")]
        if unl: flags.append(f"{len(unl)} TABLE(S) WITHOUT CAPTION LABEL")
        rows.append({"paper_id": p["paper_id"], "pages": p["num_pages"], "text": sum(c["chunk_type"] == "text" for c in pc),
                     "tables": sum(c["chunk_type"] == "table" for c in pc), "captions": sum(c["chunk_type"] == "caption" for c in pc),
                     "flags": "; ".join(flags) or "OK"})
    return rows

# =====================================================================================
# 7. RETRIEVAL: lexical normalisation (units + glossary synonyms), BM25, dense (FAISS), RRF fusion
# =====================================================================================
_SEP = r"[\s\u00b7.*()]*"
_UNIT_SUBS = [
 (r"l\s*per\s*\(?\s*m\s*\^?2\s*per\s*h(?:our)?\s*per\s*bar\s*\)?", " lmh_per_bar "),     # RSC style: L per (m2 per h per bar)
 (r"l\s*per\s*\(?\s*m\s*\^?2\s*per\s*h(?:our)?\s*\)?", " lmh "),
 (rf"l{_SEP}m{_SEP}\^?{_SEP}-?2{_SEP}h{_SEP}\^?{_SEP}-?1{_SEP}bar{_SEP}\^?{_SEP}-?1", " lmh_per_bar "),
 (rf"l\s*/\s*\(?\s*m\s*\^?2{_SEP}h{_SEP}bar\s*\)?", " lmh_per_bar "),
 (r"\blmh\s*/\s*bar\b|\blmh\s*bar\s*-?1\b", " lmh_per_bar "),
 (rf"l{_SEP}m{_SEP}\^?{_SEP}-?2{_SEP}h{_SEP}\^?{_SEP}-?1", " lmh "),
 (rf"l\s*/\s*\(?\s*m\s*\^?2{_SEP}h\s*\)?", " lmh "),
 (r"l\s*/\s*m2\s*/\s*h", " lmh "),
 (r"mg\s*/\s*l\b|mg\s*l\s*-1", " mg_l "), (r"wt\s*\.?\s*%", " wtpct "), (r"\bvol\s*\.?\s*%", " volpct "),
 (r"\bg\s*/\s*mol", " g_mol "),
]
SYNONYMS = {  # glossary: literature phrase -> canonical term appended to the text/query
 "rejection": r"remov\w*\s+(?:of\s+)?(?:up to\s+|about\s+|more than\s+|over\s+)?\d+(?:\.\d+)?\s*%|\d\s*%\s*(?:of\s+the\s+\w+\s+)?(?:was\s+|were\s+)?remov|\d\s*%\s*(?:dye\s+)?removal|retention|removal efficiency|dye removal|removal rate|separation efficiency|rejection rate|dye rejection|removal of dyes?|decolou?ri[sz]ation",
 "flux": r"permeate flux|water flux|pure water flux|permeation flux|\bpwf\b|\blmh\b",
 "permeance": r"pressure-normali[sz]ed flux|lmh_per_bar|\bgpu\b",
 "selectivity": r"separation factor",
 "deadend": r"dead-?end|unstirred|stirred cell|stirred-cell",
 "crossflow": r"cross-?flow|tangential",
 "pressure": r"\bbar\b|\bkpa\b|\bmpa\b|\bpsi\b|transmembrane pressure|applied pressure|\btmp\b",
 "concentration": r"mg_l|\bppm\b|mol/l|initial concentration",
 "psf": r"polysulfone|\budel\b", "pes": r"polyethersulfone", "pvdf": r"polyvinylidene|kynar", "pan": r"polyacrylonitrile",
 "ag": r"silver", "tio2": r"titania|titanium dioxide", "cnt": r"carbon nanotubes?|mwcnt|swcnt", "mof": r"metal-organic framework|metal organic framework",
 "zif": r"zeolitic imidazolate", "go": r"graphene oxide", "nf": r"nanofiltration", "uf": r"ultrafiltration", "ro": r"reverse osmosis",
 "mmm": r"mixed-matrix|mixed matrix", "tfc": r"thin-film composite", "tfn": r"thin-film nanocomposite",
}
_STOP = set("the of and a an in on for to with by at as is are was were be this that these those from or it its we our can which than then their".split())
def lex_text(t):
    t = norm_match(t)
    t = re.sub(r"(?<=\w)\s*-\s*(?=\d)", "-", t)
    for rx, rep in _UNIT_SUBS: t = re.sub(rx, rep, t)
    extra = [canon for canon, rx in SYNONYMS.items() if re.search(rx, t)]
    return t + " " + " ".join(extra)

def tokenize(t):
    toks = re.findall(r"\d+(?:\.\d+)?|[a-z][a-z0-9_@\-]*", lex_text(t))
    out = []
    for w in toks:
        if w in _STOP: continue
        if len(w) > 3 and w.endswith("s") and not w.endswith("ss"): w = w[:-1]
        out.append(w)
    return out

PRIOR = {"table": 0.0006, "results": 0.0003, "methods": 0.0002, "abstract": 0.0, "conclusion": -0.0002, "introduction": -0.0006, "front_matter": -0.0006, "other": -0.0002}  # tie-breakers (~2 rank positions), tuned on a small set
_QSEC = [("abstract", r"\babstract\b"), ("methods", r"\b(experimental|methods?|materials and methods)\b"),
         ("results", r"\bresults?\b"), ("conclusion", r"\bconclusions?\b"), ("introduction", r"\bintroduction\b")]
SCHEMA_QUERIES = {
  "membrane": "membrane composition base polymer polysulfone PSf PES PVDF filler MOF CNT graphene oxide loading wt% membrane thickness",
  "membrane_class": "nanofiltration ultrafiltration reverse osmosis mixed-matrix membrane thin-film composite pore size MWCO",
  "fabrication": "membrane preparation fabrication phase inversion casting interfacial polymerization solvent crosslinking post-treatment",
  "application_feed": "feed solution target dye contaminant reactive black methylene blue anionic cationic textile effluent initial concentration mg/L ppm",
  "operating_conditions": "operating pressure bar kPa transmembrane pressure applied pressure temperature pH test duration filtration time",
  "test_setup": "dead-end cross-flow stirred cell filtration setup flat sheet hollow fiber spiral wound lab-scale pilot membrane area",
  "performance": "pure water flux LMH L/m2h dye rejection removal efficiency retention % permeability permeance selectivity recovery results",
}

class HashingEmbedder:
    """Deterministic stand-in for tests ONLY (bag-of-words hashing). Not a semantic model."""
    name, dim = "hashing-test-only", 384
    def _v(self, t):
        v = np.zeros(self.dim, dtype="float32")
        for w in tokenize(t): v[zlib.crc32(w.encode()) % self.dim] += 1.0
        n = np.linalg.norm(v); return v / n if n else v
    def encode_docs(self, texts): return np.stack([self._v(t) for t in texts]) if texts else np.zeros((0, self.dim), "float32")
    def encode_query(self, q): return self._v(q)

class SentenceTransformerEmbedder:
    """Real dense model. bge-* models need an instruction prefix on QUERIES only."""
    def __init__(self, name="BAAI/bge-small-en-v1.5", fallback="sentence-transformers/all-MiniLM-L6-v2"):
        from sentence_transformers import SentenceTransformer
        try: self.model, self.name = SentenceTransformer(name), name
        except Exception as e:
            print(f"[warn] could not load {name} ({e}); falling back to {fallback}")
            self.model, self.name = SentenceTransformer(fallback), fallback
        self.prefix = "Represent this sentence for searching relevant passages: " if "bge" in self.name.lower() and "en" in self.name.lower() else ""
        self.dim = self.model.get_sentence_embedding_dimension()
    def encode_docs(self, texts): return np.asarray(self.model.encode(texts, batch_size=32, normalize_embeddings=True, show_progress_bar=len(texts) > 64), dtype="float32")
    def encode_query(self, q): return np.asarray(self.model.encode([self.prefix + q], normalize_embeddings=True), dtype="float32")[0]

def make_embedder(name="BAAI/bge-small-en-v1.5"):
    """Real dense embedder, or None (lexical-only) if sentence-transformers / the model is unavailable."""
    try: return SentenceTransformerEmbedder(name)
    except Exception as e:
        print(f"[warn] dense model unavailable ({type(e).__name__}: {e}); continuing with lexical (BM25) retrieval only"); return None

def embed_text(c):
    ctx = f"{c['title'] or ''} | {c['section']}" + (f" > {c['heading']}" if c.get("heading") else "")
    return f"{ctx}\n{c['text']}"

class Index:
    EXCLUDE = {"references", "acknowledgements"}
    def __init__(self, corpus, embedder=None):
        from rank_bm25 import BM25Okapi
        self.papers = {p["paper_id"]: p for p in corpus["papers"]}
        self.chunks, self.pages = corpus["chunks"], corpus.get("pages", {})
        self.by_id = {c["id"]: c for c in self.chunks}
        self.rows = [c["id"] for c in self.chunks if c["section"] not in self.EXCLUDE]
        self.bm25 = BM25Okapi([tokenize(self.by_id[i]["text"] + " " + self.by_id[i]["evidence_location"] + " " + " ".join(self.by_id[i]["tags"])) for i in self.rows])
        self.embedder, self.faiss, self.dense_ok = embedder, None, False
        if embedder is not None and self.rows:
            import faiss
            vec = embedder.encode_docs([embed_text(self.by_id[i]) for i in self.rows])
            self.faiss = faiss.IndexFlatIP(vec.shape[1]); self.faiss.add(vec); self.dense_ok = True

    # ---- persistence
    def save(self, folder):
        os.makedirs(folder, exist_ok=True)
        with open(f"{folder}/chunks.jsonl", "w", encoding="utf-8") as f:
            for c in self.chunks: f.write(json.dumps(c, ensure_ascii=False) + "\n")
        json.dump(list(self.papers.values()), open(f"{folder}/papers.json", "w"), indent=1, ensure_ascii=False)
        json.dump(self.pages, open(f"{folder}/pages.json", "w"), ensure_ascii=False)
        json.dump(self.rows, open(f"{folder}/row_to_chunk_id.json", "w"))
        with open(f"{folder}/tables.jsonl", "w", encoding="utf-8") as f:
            for c in self.chunks:
                if c.get("table"): f.write(json.dumps({"paper_id": c["paper_id"], "page": c["evidence_page"], **c["table"]}, ensure_ascii=False) + "\n")
        if self.dense_ok:
            import faiss; faiss.write_index(self.faiss, f"{folder}/index.faiss")
        json.dump({"embedder": getattr(self.embedder, "name", None), "dim": getattr(self.embedder, "dim", None), "num_chunks": len(self.chunks),
                   "num_indexed": len(self.rows), "schema_queries": SCHEMA_QUERIES, "dense": self.dense_ok}, open(f"{folder}/config.json", "w"), indent=1)

    @classmethod
    def load(cls, folder, embedder="auto"):
        chunks = [json.loads(l) for l in open(f"{folder}/chunks.jsonl", encoding="utf-8")]
        corpus = {"papers": json.load(open(f"{folder}/papers.json")), "chunks": chunks, "pages": json.load(open(f"{folder}/pages.json"))}
        cfg = json.load(open(f"{folder}/config.json")); self = cls(corpus, embedder=None)
        if cfg.get("dense") and os.path.exists(f"{folder}/index.faiss"):
            import faiss
            self.faiss, self.dense_ok = faiss.read_index(f"{folder}/index.faiss"), True
            self.embedder = SentenceTransformerEmbedder(cfg["embedder"]) if embedder == "auto" and "hashing" not in str(cfg["embedder"]) else (HashingEmbedder() if "hashing" in str(cfg["embedder"]) else embedder)
        return self

    # ---- search
    def _mask(self, paper_id, chunk_type, section):
        return np.array([(not paper_id or self.by_id[i]["paper_id"] == paper_id) and (not chunk_type or self.by_id[i]["chunk_type"] == chunk_type)
                         and (not section or self.by_id[i]["section"] == section) for i in self.rows])

    def search(self, query, k=5, paper_id=None, chunk_type=None, section=None, mode="hybrid", return_all=False, prior=True):
        if not self.rows: return []
        mask = self._mask(paper_id, chunk_type, section)
        if not mask.any(): return []
        ranks = []
        if mode in ("lexical", "hybrid"):
            s = np.array(self.bm25.get_scores(tokenize(query))); s = np.where(mask, s, -1e9)
            order = [int(i) for i in np.argsort(-s) if mask[i] and s[i] > 0]; ranks.append(("lex", order, s))
        if mode in ("dense", "hybrid") and self.dense_ok:
            q = self.embedder.encode_query(query).astype("float32")[None, :]
            sc, ix = self.faiss.search(q, len(self.rows)); d = np.full(len(self.rows), -1e9); d[ix[0]] = sc[0]; d = np.where(mask, d, -1e9)
            order = [int(i) for i in np.argsort(-d) if mask[i]]; ranks.append(("dense", order, d))
        if not ranks: return []
        fused = {}
        for _, order, _s in ranks:
            for r, i in enumerate(order[:100]): fused[i] = fused.get(i, 0) + 1.0 / (60 + r)
        if prior:   # evidence-quality prior: own results (tables/results/methods) beat intro/conclusion/front-matter (other papers' numbers)
            for i in fused:
                c = self.by_id[self.rows[i]]
                fused[i] += PRIOR.get("table" if c["chunk_type"] == "table" else c["section"], 0.0)
        want = [g for g, rx in _QSEC if re.search(rx, query, re.I)]
        if want:
            for i in fused:
                if self.by_id[self.rows[i]]["section"] in want: fused[i] += 0.012
        top = sorted(fused, key=lambda i: -fused[i])
        out = []
        for i in (top if return_all else top[:k]):
            c = dict(self.by_id[self.rows[i]]); c["score"] = round(fused[i] if mode == "hybrid" else float(dict((n, s) for n, _, s in ranks).get("lex" if mode == "lexical" else "dense")[i]), 4)
            out.append(c)
        return out

    def retrieve_for_paper(self, pid, k=4):
        ctx = {}
        for sec, q in SCHEMA_QUERIES.items():
            hits = self.search(q, k, pid) + self.search(q, 6 if sec == "performance" else 2, pid, chunk_type="table")
            if sec in ("performance", "operating_conditions"): hits += self.search(q, 1, pid, section="abstract")
            seen, merged = set(), []
            for h in hits:
                if h["id"] not in seen: seen.add(h["id"]); merged.append(h)
            ctx[sec] = merged
        return ctx

    MIN_SNIPPET_CHARS = 12
    def verify_snippet(self, paper_id, page, snippet):
        """True only if the snippet appears (whitespace/dash/quote-insensitive) on that exact page.
        Empty / very short snippets are rejected (they would match almost anything). A snippet containing an ellipsis
        ('...', unicode ellipsis or '[...]') is verified fragment by fragment, in order."""
        pg = self.pages.get(paper_id, {})
        text = pg.get(page, pg.get(str(page), ""))
        if not text or not isinstance(snippet, str): return False
        frags = [norm_match(f) for f in re.split(r"\[\s*\.\.\.\s*\]|\.{3,}|\u2026", snippet)]
        frags = [f for f in frags if f]
        if not frags or any(len(f) < self.MIN_SNIPPET_CHARS for f in frags): return False
        pos = 0
        for f in frags:
            k = text.find(f, pos)
            if k < 0: return False
            pos = k + len(f)
        return True

# =====================================================================================
# 7b. OPTIONAL METADATA ENRICHMENT (Crossref) -- fixes the heuristic authors/journal fields
# =====================================================================================
def _apply_crossref(paper, msg):
    """Pure function: copy title/authors/journal/year from a Crossref 'message' dict into a paper record. Returns the fields changed."""
    changed = {}
    t = (msg.get("title") or [None])[0]
    au = ", ".join(" ".join(x for x in (a.get("given"), a.get("family")) if x) for a in msg.get("author", []) if a.get("family") or a.get("name"))
    if not au and msg.get("author"): au = ", ".join(a.get("name", "") for a in msg["author"] if a.get("name"))
    j = (msg.get("container-title") or [None])[0]
    yr = None
    for k in ("published-print", "published-online", "published", "issued"):
        dp = (msg.get(k) or {}).get("date-parts") or [[None]]
        if dp and dp[0] and dp[0][0]: yr = int(dp[0][0]); break
    for key, val in (("title", t), ("authors", au or None), ("journal", j), ("publication_year", yr)):
        if val and paper.get(key) != val: changed[key] = (paper.get(key), val); paper[key] = val
    if changed:
        paper["metadata_source"] = "crossref"
        paper["needs_review"] = [x for x in paper.get("needs_review", []) if x not in changed and not x.startswith("authors/journal")]
    return changed

def enrich_metadata_crossref(corpus, timeout=10, mailto=None):
    """For every paper with a DOI, fetch Crossref metadata (needs internet). Failures are reported, never fatal.
    Check the result against the Ground Truth 'Papers' sheet -- Crossref can differ from the PDF (e.g. online vs print year)."""
    import urllib.request, urllib.parse
    report = []
    for p in corpus["papers"]:
        if not p.get("doi"): report.append((p["paper_id"], "no DOI detected")); continue
        url = "https://api.crossref.org/works/" + urllib.parse.quote(p["doi"], safe="/") + (f"?mailto={mailto}" if mailto else "")
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "membrane-index/1.0"}), timeout=timeout) as r:
                msg = json.load(r)["message"]
            ch = _apply_crossref(p, msg); report.append((p["paper_id"], ("updated: " + ", ".join(ch)) if ch else "no change"))
        except Exception as e:
            report.append((p["paper_id"], f"crossref failed: {type(e).__name__}"))
    return report

# =====================================================================================
# 8. EVALUATION AGAINST GROUND TRUTH
# =====================================================================================
def _match(c, g):
    """g['loc'] may be one location or a list of acceptable ones (e.g. the table OR the results sentence)."""
    if c["paper_id"] != g["paper_id"]: return False
    if g.get("page") is not None and c["evidence_page"] != g["page"]: return False
    locs = g.get("loc", "")
    for loc in (locs if isinstance(locs, list) else [locs]):
        if loc.startswith(("Table", "Figure")):
            if c["evidence_location"].startswith(loc): return True
        elif loc.startswith("Section:"):
            if c["section"] == loc.split(":", 1)[1].strip(): return True
        else: return True
    return False

def evaluate(index, gt, modes=("lexical", "dense", "hybrid"), scoped=True, ks=(1, 3, 5)):
    res = {}
    for mode in modes:
        if mode in ("dense", "hybrid") and not index.dense_ok: continue
        ranks = []
        for g in gt:
            hits = index.search(g["q"], max(ks), paper_id=g["paper_id"] if scoped else None, mode=mode)
            r = next((i + 1 for i, h in enumerate(hits) if _match(h, g)), None); ranks.append(r)
        n = len(ranks)
        res[mode] = {**{f"R@{k}": round(sum(1 for r in ranks if r and r <= k) / n, 3) for k in ks}, "MRR": round(sum(1 / r for r in ranks if r) / n, 3), "ranks": ranks}
    return res


def context_recall(index, gt_rows, k=4):
    """Share of Ground-Truth evidence rows (paper_id, page, loc) that appear in retrieve_for_paper() context.
    This is what the extraction step actually sees, so a miss here = guaranteed extraction miss."""
    ctx, hit, miss = {}, 0, []
    for g in gt_rows:
        pid = g["paper_id"]
        if pid not in ctx: ctx[pid] = [c for sec in index.retrieve_for_paper(pid, k).values() for c in sec]
        if any(_match(c, g) for c in ctx[pid]): hit += 1
        else: miss.append(g)
    return {"recall": round(hit / max(len(gt_rows), 1), 3), "n": len(gt_rows), "misses": miss}

def load_gt_evidence(xlsx_path, sheet="Evidence"):
    """Read Ground_Truth_Master.xlsx -> [{paper_id, page, loc}]. Column names are matched loosely."""
    import pandas as pd
    df = pd.read_excel(xlsx_path, sheet_name=sheet); cols = {c.lower().strip(): c for c in df.columns}
    pick = lambda *keys: next((v for k, v in cols.items() if all(x in k for x in keys)), None)
    pc, gc, lc = pick("paper", "id") or pick("paper"), pick("page"), pick("location") or pick("table")
    rows = []
    for _, r in df.iterrows():
        m = re.search(r"\d+", str(r[gc])) if gc else None
        loc = str(r[lc]) if lc is not None and str(r[lc]) != "nan" else ""
        mt = re.search(r"(Table|Figure|Fig\.?)\s*(S?\d+)", loc, re.I)
        rows.append({"paper_id": str(r[pc]).strip().upper(), "page": int(m.group(0)) if m else None,
                     "loc": (("Table " if mt.group(1).lower().startswith("t") else "Figure ") + mt.group(2)) if mt else ""})
    return [r for r in rows if r["paper_id"] and r["paper_id"] != "NAN"]


## 1. Upload the PDFs

In [ ]:
import os
from google.colab import files
os.makedirs("papers", exist_ok=True)
up = files.upload()
for n, d in up.items(): open(os.path.join("papers", n), "wb").write(d)
print(sorted(os.listdir("papers")))   # expect P01.pdf, P02.pdf, ...

## 2. Parse + health check
Fix or replace any paper that is not `OK` (scanned PDF, image tables, missing sections) **before** the demo.

In [ ]:
import importlib, pandas as pd
import membrane_index as mi
importlib.reload(mi)
pd.set_option("display.max_colwidth", 90, "display.width", 220)
corpus = mi.build_corpus("papers")
display(pd.DataFrame(corpus["papers"])[["paper_id", "title", "authors", "publication_year", "journal", "doi", "num_pages"]])
display(pd.DataFrame(mi.parse_health(corpus)))

## 2b. (Optional, needs internet) Fill authors / journal / year from Crossref
The PDF heuristics for authors and journal are unreliable. This uses each paper's DOI. **Check the result against the Ground Truth `Papers` sheet** (Crossref can give the online year instead of the print year).

In [ ]:
for pid, msg in mi.enrich_metadata_crossref(corpus): print(pid, msg)
display(pd.DataFrame(corpus["papers"])[["paper_id", "title", "authors", "publication_year", "journal", "doi"]])

## 3. Spot-check the tables against the PDFs
Tables are the main data source. Open each PDF next to this output and confirm headers and a few rows.

In [ ]:
for ch in corpus["chunks"]:
    if ch["chunk_type"] == "table":
        t = ch["table"]
        print(f"\n{ch['paper_id']}  p.{ch['evidence_page']}  {t['label']}  ({t['detector']}{', continued' if t['continued'] else ''})")
        print("  header:", t["header"])
        for r in t["rows"][:3]: print("  row   :", r)

## 4. Build the index
Uses `BAAI/bge-small-en-v1.5` for the dense part (falls back to lexical-only if it cannot load).

In [ ]:
emb = mi.make_embedder("BAAI/bge-small-en-v1.5")      # alternative: "sentence-transformers/all-MiniLM-L6-v2"
idx = mi.Index(corpus, embedder=emb)
print("dense retrieval:", idx.dense_ok, "| indexed chunks:", len(idx.rows), "of", len(idx.chunks))

## 5. (Optional) Check against the Ground Truth workbook
`context_recall` = share of Ground-Truth evidence rows (page + table) that appear in the context the extractor receives. A miss here is a guaranteed extraction miss.
Skip by cancelling the upload dialog.

In [ ]:
up = files.upload()   # Ground_Truth_Master.xlsx
if up:
    name = list(up)[0]; open(name, "wb").write(up[name])
    gt = mi.load_gt_evidence(name)          # reads sheet "Evidence"; edit sheet= if yours is named differently
    r = mi.context_recall(idx, gt)
    print("context recall:", r["recall"], "of", r["n"], "evidence rows")
    for m in r["misses"][:25]: print("MISS", m)

## 6. Compare lexical / dense / hybrid on YOUR papers
Edit `QUERIES`: 10–20 realistic questions, with where the answer really is (`Table N` and/or `Section: methods|results|abstract`). Pick the mode with the best R@3 / MRR.

In [ ]:
QUERIES = [   # EDIT THESE
 dict(paper_id="P01", q="pure water flux of the best membrane", loc=["Table 2", "Section: results"]),
 dict(paper_id="P01", q="operating pressure and temperature of the filtration test", loc="Section: methods"),
]
res = mi.evaluate(idx, QUERIES)
for mode, r in res.items(): print(mode, {k: v for k, v in r.items() if k != "ranks"})

## 7. Save and download

In [ ]:
idx.save("faiss_out")
!zip -qr faiss_out.zip faiss_out
files.download("faiss_out.zip")

## 8. Try it
`verify_snippet` is the hallucination guard: it is True only if the quoted text really appears on that page.

In [ ]:
for h in idx.search("methylene blue rejection at 4 bar", k=3):
    print(h["paper_id"], "p.%s" % h["evidence_page"], h["evidence_location"], "|", h["text"][:100].replace("\n", " "))
ctx = idx.retrieve_for_paper(corpus["papers"][0]["paper_id"])
print({k: len(v) for k, v in ctx.items()})
# idx.verify_snippet("P01", 5, "pure water flux of 89 LMH")

## 9. Export for the Streamlit app
Run this **after step 4** (and after the optional Crossref step 2b, so titles/years are included).

What it does:
- takes the vectors straight out of the FAISS index you just built, so **row *i* of `embeddings.npy` = line *i* of `chunks.jsonl`**;
- keeps your evidence fields (`evidence_location`, parsed `table`, `tags`, `section`, `heading`) and uses the paper id (`P01`) as the app's `source`;
- saves the query prefix of your embedding model, so the app embeds questions exactly like your notebook;
- saves `pages.json` (normalised page text) so the app can verify quoted evidence and numbers on the exact page.

Put the downloaded files into the repo folder `data/imported/`.

In [ ]:
import json, os, shutil
import numpy as np

assert idx.dense_ok and emb is not None, "Dense index missing: re-run step 4 until it prints 'dense retrieval: True'."
assert "hashing" not in str(getattr(emb, "name", "")), "The test-only hashing embedder cannot be used for the app."

APP_DIR = "app_files"
os.makedirs(APP_DIR, exist_ok=True)

# 1) vectors in FAISS row order (idx.rows = chunk ids that were indexed; references/acknowledgements are excluded)
vectors = np.asarray(idx.faiss.reconstruct_n(0, idx.faiss.ntotal), dtype="float32")
assert vectors.shape[0] == len(idx.rows), "FAISS rows and indexed chunks differ"

# 2) chunks in the same order, in the format the app reads
chunk_rows = []
for cid in idx.rows:
    c = idx.by_id[cid]
    p = idx.papers[c["paper_id"]]            # paper-level info (includes Crossref updates if you ran step 2b)
    chunk_rows.append({
        "chunk_id": f"{c['paper_id']}_c{c['id']}",
        "text": c["text"],
        "source": c["paper_id"],             # P01, P02, ... (matches your Ground Truth ids)
        "page": c["evidence_page"],
        "section": c["section"],
        "heading": c.get("heading", ""),
        "chunk_type": c["chunk_type"],       # text | table | caption
        "evidence_location": c["evidence_location"],
        "table": c.get("table"),
        "tags": c.get("tags", []),
        "title": p.get("title"), "doi": p.get("doi"),
        "publication_year": p.get("publication_year"), "file": p.get("file"),
    })

# 3) the same checks the app runs when it loads the files
ids = [r["chunk_id"] for r in chunk_rows]
assert len(ids) == len(set(ids)), "chunk_id must be unique"
assert all(r["text"].strip() for r in chunk_rows), "empty chunk text found"
assert all(isinstance(r["page"], int) for r in chunk_rows), "page must be an integer"
assert vectors.ndim == 2 and len(chunk_rows) == vectors.shape[0], "row count mismatch"
assert np.isfinite(vectors).all(), "NaN/inf in embeddings"

# 4) write the files
with open(f"{APP_DIR}/chunks.jsonl", "w", encoding="utf-8") as f:
    for r in chunk_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
np.save(f"{APP_DIR}/embeddings.npy", vectors)
json.dump({"model": emb.name, "dimension": int(vectors.shape[1]), "normalized": True,
           "query_prefix": getattr(emb, "prefix", "")},
          open(f"{APP_DIR}/embedding_info.json", "w"), indent=2)
json.dump(idx.pages, open(f"{APP_DIR}/pages.json", "w"), ensure_ascii=False)
json.dump(list(idx.papers.values()), open(f"{APP_DIR}/papers.json", "w"), ensure_ascii=False, indent=1)

print(f"OK  {len(chunk_rows)} chunks x {vectors.shape[1]} dims | model: {emb.name}")
print(f"OK  papers: {len(idx.papers)} | text chunks: {sum(r['chunk_type']=='text' for r in chunk_rows)} | "
      f"table chunks: {sum(r['chunk_type']=='table' for r in chunk_rows)} | captions: {sum(r['chunk_type']=='caption' for r in chunk_rows)}")
for n in sorted(os.listdir(APP_DIR)):
    print(f"   {n:22s} {os.path.getsize(os.path.join(APP_DIR, n)) / 1e6:6.2f} MB")

shutil.make_archive("app_files", "zip", APP_DIR)
from google.colab import files
files.download("app_files.zip")

### After the download
1. Unzip `app_files.zip` and copy **all files** into the repo folder `data/imported/` (`papers.json` is optional, the rest are used by the app).
2. Commit and push, then reboot the Streamlit app.
3. Open **Data check**: you should see your chunk and paper counts, titles, "Page text: loaded", and **Test embedding model** should report that the dimension matches.

**Adding or replacing a PDF later:** re-run the whole notebook with all PDFs (so `P01..Pn` stay aligned) and replace these files.